# 2. PANDAS

## 2.1 Series and Dataframe

A step more from NumPy, Pandas offers a structural and database-like choice, which can label the features. We can view it as an Excel that can be manipulated by Python.


In [ ]:
import pandas as pd

scores = pd.Series(
    [80, 90, 70],
    index=["Alice", "Bob", "Charlie"],
)

print(scores)

# The output should be:
# Alice      80
# Bob        90
# Charlie    70
# dtype: int64

### 2.1.1 Series

Series is a one-dimentional data that contains index (or label).

Thus, Pandas allows us to access the data by referring the index. E.g:


In [ ]:
print(scores["Bob"])  # 90

# However, notice that `scores` is still one dimentional:
print(scores.shape)  # (3,)
print(scores.ndim)  # 1

### 2.1.2 DataFrame

DataFrame is two-dimentional, which can be obtained from reshaping the series:


In [ ]:
df = pd.DataFrame(
    {
        "Name": ["Alice", "Bob", "Charlie"],
        "Math": [80, 85, 60],
        "English": [90, 95, 75],
        "Python": [70, 90, 80],
    },
    index=["S001", "S002", "S003"],
)

print(df)

print(df.shape)  # (3, 4)
print(df.columns)  # column label
print(df.index)  # row index
print(df.dtypes)  # datatype of each column

In the code above:

- `Name`, `Math`, `English` and `Python` are column index;
- `S001`, `S002`, `S003` are row index.

Then how can we select data from the dataframe? There are basically two types:

1. Selecting by the label of column:


In [ ]:
math_scores = df["Math"]  # The Index is saved, and We get a series
print(type(math_scores))

math_scores = df[
    ["Math"]
]  # We get a dataframe, and `Math` is a column of the dataframe
print(type(math_scores))

2. Selecting by calling functions.

`.loc` and `iloc` can both select row and column, but the difference is that:

- `.loc` uses labels, while `.iloc` uses index (integers).
- `.iloc` can use `a : b` to select a subdataframe, but not including the `b`, however `.loc` will often include the right parameter.


In [ ]:
df.loc["S002", "English"]
df.iloc[1, 2]

# df.loc[row_label, column_label]
# df.iloc[row_position, column_position]

df.iloc[0:2, 1:3]
df.loc["S001":"S002", "Math":"English"]  # including row S002 and column English.

### 2.1.3 Boolean Indexing

Similar to NumPy, Pandas also allow boolean indexing.


In [ ]:
condition = df["Math"] >= 80
selected = df[condition]
print(selected)

# We can also select part of the subdataframe:
selected = df.loc[
    df["Math"] >= 80,  # row condition as index
    ["Name", "Math"],  # column index
]
print(selected)

## 2.2 CSV Reading and Data Analysis

### 2.2.1 Things About CSV

CSV is the abbreviation of 'Comma-Seerated-Values', which stores dataframe in pure text. For example:

        Name,Math,English,Python
        Alice,80,90,70
        Bob,85,95,90
        Charlie,60,75,80

=> The first line is often Header.


In [ ]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "students.csv"

csv_path.write_text(
    """StudentID,Class,Name,Math,English,Python
S001,A,Alice,80,90,70
S002,A,Bob,85,95,90
S003,B,Charlie,60,75,80
S004,B,Diana,90,,95
S005,A,Eva,not_recorded,88,92
S006,B,Frank,75,85,80
""",
    encoding="utf-8",
)

# This code creates and writes a csv file in the current directory /data.
# However, the English of Diana is missing, and the Math of Eva is not_recorded.

After creating the csv file, we can read it with function `.read_csv()`:


In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
display(df)

print(df.iloc[1, 3])

Some parameters of `.read_csv()`:

- `header = 0`: set the first row as label (default);
- `header = None`: don't set any line as label;
- `sep = ","`: set the segment label;
- `index_col = "StudentID"`: set a column to be row index;
- `encoding = "utf-8"`: set the encoding of text.


### 2.2.2 Data Inspection

After reading a new database, we should not immediately start training models. It's often suggested to use:


In [ ]:
print(df.shape)
print(df.dtypes)

display(df.head())
display(df.describe())

Pandas also offers a great deal of functions:

- `.head(n)`: display the first n row, while the default value of n is 5;
- `.tail(n)`: display the last n row;
- `.info()`: inspect the type of column, memory, and all that jazz;
- `.describe()`: see the description statistics of the values;
- `.types()`: see the type of data.


### 2.2.3 Data Cleaning

Pandas uses `NaN`, `None`, `pd.NA`, etc. to indicate data loss in different situations. For example, we can use:


In [ ]:
print(df.isna())  # check through every cell if there is data loss

print(df.isna().sum())  # check the sum of loss data

`.isna()` returns a boolean value, and `True` indicates a data loss in the cell.

However, the "not_recorded" in Math is detected as a string, thus no data loss in that cell is reported.

In this case, we have to do a type change:


In [ ]:
df["Math"] = pd.to_numeric(  # to_numeric tries to cast the data to numbers.
    df["Math"],
    errors="coerce",  # data that cannot be casted will not assert an error, but turn to the loss value.
)

then how can we handle the loss data? We often use the two methods bellow:

1. Delete the record: the `.dropna()` function deletes the whole row with data loss.


In [ ]:
df_dropped = df.dropna()
print(df_dropped)

2. Complete the data by using `.fillna()`.


In [ ]:
subjects = ["Math", "English", "Python"]

for col in subjects:
    df[col] = df[col].fillna(
        df[col].median()
    )  # use the median of each course to fill the missing cells, and then cover the old dataframe with the new one.

### 2.2.4 Descriptive Statistics

After data cleaning, we can inspect statistics of the values. The `.describe()` function will give multiple usefull statstics, including count, mean, std, min, max, etc.


In [ ]:
# Adding average column
df["Average"] = df[subjects].mean(axis=1)

# using groups to categorize
# suppose we want to compare the average grade of class A and B
class_average = df.groupby("Class")["Average"].mean()
print(class_average)

# calculating multiple values in a single statement:
summary = df.groupby("Class")["Average"].agg(["count", "mean", "max"])
display(summary)

# saving the new dataframe to a csv file:
df.to_csv(
    data_dir / "students_cleaned.csv",
    index=False,  # do NOT use the current row index as an extra column in the new file.
    encoding="utf-8",
)